# 🔍 Initial Data Verification
This notebook performs an initial check of the data, identifying useful variables, data quality, and adjustment needs. It generates a sampled dataset for POC/Ad-hoc analysis.


## 1. Libraries

In [ ]:
# Utilities
import pandas as pd
import numpy as np
import datetime
import os
import warnings

# Spark Functions
from pyspark.sql.functions import col, when, lit

# Calculations
from scipy import stats

# Graphics
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook Config
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid", palette="pastel")
warnings.filterwarnings('ignore')


## 2. Import and Initial Preparation

In [ ]:
# =============================================================================
# S&OP Data - Product List
# =============================================================================
# Path to folder with latest file
sop_dir = '/dbfs/mnt/operations/sop_data/'

# Get latest .xlsx file
latest_file = max([os.path.join(sop_dir, f) for f in os.listdir(sop_dir) if f.endswith('.xlsx')], key=os.path.getmtime)

df_sop = pd.read_excel(latest_file, sheet_name='MASTER_DATA')
df_sop['EAN_CODE'] = df_sop['EAN_CODE'].astype(str).str.strip()

# Check file modification date
sop_info = os.stat(latest_file)
dt_mod_sop = datetime.datetime.fromtimestamp(sop_info.st_mtime).strftime('%d-%m-%Y %H:%M')
print(f"Last S&OP file modification: {dt_mod_sop}")

# BUs to consider
bu_list = ['BU_A', 'BU_B', 'BU_C', 'BU_D']

# Filter EANs
eans_list = df_sop[(df_sop['CONSOLIDATED_BU'].isin(bu_list)) & ~(df_sop['EAN_CODE'].isin(['#_NA']))]['EAN_CODE'].unique().tolist()


# =============================================================================
# Final Orders Data
# =============================================================================

# Columns
cols_sales   = ['CUSTOMER_TAX_ID', 'EAN_CODE', 'ORDER_DATE', 'BUSINESS_UNIT', 'SUB_BU', 'BRAND']
cols_metrics = ['BILLED_QTY', 'FACTORY_PRICE_TOTAL', 'LIST_PRICE_TOTAL', 'ORDER_DISCOUNT_VAL']
cols_channel = ['CUSTOMER_ID', 'REGION_STATE', 'SUB_CHANNEL', 'SALES_TEAM_CHANNEL', 'TEAM_NAME', 'BRAND_CATEGORY']

# Parameterizable list of teams
team_list = ['TEAM_1', 'TEAM_2', 'TEAM_3']
team_str  = ",".join([f"'{c}'" for c in team_list])

# EANs string for SQL
eans_str_sql = ",".join([f"'{ean}'" for ean in eans_list])

# Explicit selects
select_sales = ",\n       ".join(cols_sales + cols_metrics)
select_channel = ", ".join([f"ch.{c}" for c in cols_channel if c != 'CUSTOMER_ID'])

# Fixed sample size
sample_size = 1000000 # Approx 4% of data

query = f"""
-- CTE 'orders': filters orders by EAN and DATE, calculates period for join
WITH orders AS (
  SELECT
    {select_sales
},
    -- convert ORDER_DATE to int YYYYMM
    CAST(date_format(ORDER_DATE, 'yyyyMM') AS INT) AS order_period_key
  FROM sales_db.fact_sales_refined
  WHERE
    EAN_CODE IN ({eans_str_sql
})
    AND ORDER_DATE >= '2023-01-01'),

-- CTE 'channel_ranked': links orders to channel history and ranks by recency
channel_ranked AS (
  SELECT
    ch.*,
    o.order_period_key,
    ROW_NUMBER() OVER (
      PARTITION BY ch.CUSTOMER_ID, o.order_period_key
      ORDER BY ch.PERIOD_KEY DESC) AS rn
  FROM sales_db.dim_channel_panel ch
  JOIN orders o
    ON ch.CUSTOMER_ID = o.CUSTOMER_TAX_ID
   AND ch.PERIOD_KEY <= o.order_period_key),

-- CTE 'channel_selected': keeps only most recent record (rn = 1)
channel_selected AS (
  SELECT
    CUSTOMER_ID,
    order_period_key,
{
  ", ".join(cols_channel[
    1:])
}
  FROM channel_ranked
  WHERE rn = 1),

-- CTE 'cogs_total': gets FINAL_UNIT_VALUE for COGS - Total per EAN/Period
cogs_total AS (
  SELECT
    EAN_CODE,
    CAST(YEAR_MONTH AS INT) AS cogs_period_key,
    FINAL_UNIT_VALUE AS unit_cogs_total
  FROM finance_db.ref_unit_cogs_hist
  WHERE ACCOUNT_CATEGORY = 'COGS - Total'),

-- CTE 'joined': joins orders, channel and COGS
joined AS (
  SELECT
    ord.EAN_CODE,
    ord.ORDER_DATE,
    ord.BUSINESS_UNIT,
    ord.SUB_BU,
    ord.BRAND,
    ord.BILLED_QTY,
    ord.FACTORY_PRICE_TOTAL,
    ord.LIST_PRICE_TOTAL,
    ord.ORDER_DISCOUNT_VAL,
{select_channel
},
    cost.unit_cogs_total
  FROM orders ord
  LEFT JOIN channel_selected ch
    ON ord.CUSTOMER_TAX_ID = ch.CUSTOMER_ID
   AND ord.order_period_key = ch.order_period_key
  LEFT JOIN cogs_total cost
    ON ord.EAN_CODE = cost.EAN_CODE
   AND ord.order_period_key = cost.cogs_period_key
  WHERE
    ch.TEAM_NAME IN ({team_str
})
    AND ch.SUB_CHANNEL IS NOT NULL
    AND ord.BILLED_QTY > 0)

-- Select all and shuffle for sampling
SELECT *
FROM joined
ORDER BY rand()
LIMIT {sample_size
}
"""

'''
Data without sampling: ~23M rows
'''
# Execute Query
df_raw = spark.sql(query).toPandas()


Total records without sampling (Jan 2023 onwards) is approx 15.6M rows.

In [ ]:
# 1. Dictionary mapping Region to States
region_states = {
    'North': ['AC','AP','AM','PA','RO','RR','TO'  ],
    'Northeast': ['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
    'Midwest': ['MT','MS','GO','DF'],
    'Southeast': ['SP','RJ','ES','MG'],
    'South': ['PR','SC','RS']
}

# 2. Inverse dictionary mapping State -> Region
state_to_region = {
    state: region
    for region, states in region_states.items()
    for state in states
}

# 4. Create 'REGION' column
df_raw['REGION'] = df_raw['REGION_STATE'].map(state_to_region)

In [ ]:
# Adjust Date formats
df_raw['ORDER_DATE'] = pd.to_datetime(df_raw['ORDER_DATE'], format='%Y-%m-%d')

# Adjust Numeric formats
df_raw['BILLED_QTY'] = df_raw['BILLED_QTY'].astype(int)
df_raw['FACTORY_PRICE_TOTAL'] = df_raw['FACTORY_PRICE_TOTAL'].astype(float)
df_raw['LIST_PRICE_TOTAL'] = df_raw['LIST_PRICE_TOTAL'].astype(float)
df_raw['ORDER_DISCOUNT_VAL'] = df_raw['ORDER_DISCOUNT_VAL'].astype(float)

# Adjust Categorical formats
df_raw['BRAND_CATEGORY'] = df_raw['BRAND_CATEGORY'].astype(str)

In [ ]:
# Unit Metrics
df_raw['FACTORY_PRICE_UNIT'] = df_raw['FACTORY_PRICE_TOTAL'] / df_raw['BILLED_QTY']
df_raw['NET_PRICE_UNIT'] = df_raw['LIST_PRICE_TOTAL'] / df_raw['BILLED_QTY']

# Adjust discount scale
df_raw[['ORDER_DISCOUNT_VAL']] = df_raw[['ORDER_DISCOUNT_VAL']] / 100

In [ ]:
df_raw.info()

In [ ]:
# =============================================================================
# Calculating Real Unit Margin
# =============================================================================

# Real Margin using COGS
df_raw['REAL_UNIT_MARGIN'] = (df_raw['NET_PRICE_UNIT'] - df_raw['unit_cogs_total'].abs()) / df_raw['NET_PRICE_UNIT']

# Theoretical Margin using Factory Price
df_raw['THEORETICAL_UNIT_MARGIN'] = (df_raw['FACTORY_PRICE_UNIT'] - df_raw['unit_cogs_total'].abs()) / df_raw['FACTORY_PRICE_UNIT']

In [ ]:
# Date Components
df_raw['month'~] = df_raw['ORDER_DATE'].dt.month
df_raw['day_of_week'] = df_raw['ORDER_DATE'].dt.dayofweek
df_raw['week_of_year'] = df_raw['ORDER_DATE'].dt.isocalendar().week.astype(int)

In [ ]:
display(df_raw)

In [ ]:
df_raw.info()


## 3. Analysis

In [ ]:
display(df_raw)

In [ ]:
df_raw['THEORETICAL_UNIT_MARGIN'].describe()

In [ ]:
df_raw['REAL_UNIT_MARGIN'].describe()

In [ ]:
df_raw['unit_cogs_total'].describe()

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df_raw,
    title="POC Data Profile - Smart Pricing (Before Processing)",
    explorative=True)

profile.to_file('/dbfs/mnt/sandbox/reports/data_profile_before_processing.html')

In [ ]:
displayHTML("""
  <a href="files/sandbox/reports/data_profile_before_processing.html"
     download="data_profile_before_processing.html"
     style="font-size:16px;">
    ⬇️ Click here to download data profile (HTML)
  </a>
""")

In [ ]:
'''
Notes:
Production cost per SKU?
Planning/Supply check?
Contact Finance Team
'''

In [ ]:
# Calculate z-scores for all numeric columns
num_cols = df_raw.select_dtypes(include='number').columns
df_raw[[f'{c}_zscore' for c in num_cols]] = df_raw[num_cols].apply(stats.zscore, nan_policy='omit')

In [ ]:
# Outlier threshold
z_threshold = 3

num_cols = df_raw.select_dtypes(include='number').columns

# Calculate outlier percentage
outlier_stats = []
total = len(df_raw)

for col in num_cols:
    zcol = f'{col}_zscore'
    if zcol not in df_raw:
        continue
    n_out = (df_raw[zcol].abs() > z_threshold).sum()
    pct = n_out / total * 100
    outlier_stats.append({
        'variable': col,
        'outliers': n_out,
        '% of total': pct
})

outliers_df = pd.DataFrame(outlier_stats) \
    .sort_values('% of total', ascending=False) \
    .reset_index(drop=True)

print(f"Total rows: {total}\n")
print(outliers_df.to_string(index=False))

In [ ]:
df_raw.describe()

In [ ]:
df_raw.drop(columns=df_raw.filter(regex='zscore').columns, inplace=True)

In [ ]:
# Saving data
df_raw_spark = spark.createDataFrame(df_raw)
mode = 'overwrite'
overwriteSchema = 'True'
table_name = 'sandbox_db.smart_pricing_flat_table_sampled_raw'

df_raw_spark.write.option("overwriteSchema", overwriteSchema).saveAsTable(table_name, 
                                                                          format='delta', 
                                                                          mode=mode)